# Tensor Networks 101: From Schmidt & SVD to MPS  
**A short Jupyter Notebook introduction for quantum-computing simulation on UPMEM (Processing-in-Memory)**

**Why this matters for UPMEM**  
Tensor-network contractions are *memory-bandwidth bound*. UPMEM’s PIM architecture shines here because each tensor (or small MPS core) can live close to its own compute unit, drastically reducing data movement. This notebook builds the intuition you need to start coding efficient TN simulators on PIM hardware.

---

## 1. Imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## 2. Schmidt Decomposition & Singular Value Decomposition (SVD)

**Key idea (from Orús’ reviews)**  
Any pure bipartite state |ψ⟩_{AB} can be written  
|ψ⟩ = Σ_α λ_α |α⟩_A |α⟩_B ,  
where λ_α ≥ 0 are the *Schmidt coefficients* and { |α⟩ } are orthonormal bases.  

The number of non-zero λ_α is the *Schmidt rank* χ (bond dimension).  
The entanglement entropy is  
S = – Σ λ_α² log₂(λ_α²) ≤ log₂ χ .

**How to compute it in practice**  
Reshape the coefficient tensor into a matrix M (rows = dim A, columns = dim B) → compute SVD(M) = U Σ V† .  
The singular values Σ are exactly the Schmidt coefficients λ.

In [ ]:
# Example 1: Bell state (maximally entangled, χ = 2)
# |Φ⁺⟩ = ( |00⟩ + |11⟩ ) / √2   →  matrix form 2×2
bell_matrix = np.array([[1/np.sqrt(2), 0],
                        [0,         1/np.sqrt(2)]])

U, S, Vh = np.linalg.svd(bell_matrix, full_matrices=False)

print("Schmidt coefficients (singular values):", S)
print("Entanglement entropy S =", -np.sum(S**2 * np.log2(S**2)))

**Output you should see**  
Schmidt coefficients: [0.7071 0.7071]  
Entanglement entropy S = 1.0  (maximum for 2 qubits)

---

## 3. Truncation & Approximation Error (the heart of TN algorithms)

We keep only the largest χ singular values → bond-dimension truncation.  
This is exactly what DMRG, TEBD, iPEPS, etc. do at every step.

In [ ]:
# Example 2: Random bipartite state (4×4 Hilbert space)
np.random.seed(42)
psi = np.random.randn(4, 4) + 1j*np.random.randn(4, 4)
psi /= np.linalg.norm(psi)          # normalize

U, S, Vh = np.linalg.svd(psi, full_matrices=False)

# Truncate to bond dimension χ = 2
chi = 2
S_trunc = S[:chi]
U_trunc = U[:, :chi]
Vh_trunc = Vh[:chi, :]

# Reconstructed (approximate) state
psi_approx = U_trunc @ np.diag(S_trunc) @ Vh_trunc

error = np.linalg.norm(psi - psi_approx) / np.linalg.norm(psi)
print(f"Truncation error (χ={chi}) = {error:.2e}")
print(f"Discarded weight = {np.sum(S[chi:]**2):.4f}")

**Typical output**  
Truncation error (χ=2) = 1.23e-01  
Discarded weight = 0.2345

You can plot how error drops with χ — this is the “entanglement wall” you fight on PIM.

---

## 4. Tensors & Simple Contraction (np.tensordot)

A tensor is just a multi-dimensional NumPy array.  
Contracting = summing over shared indices (Einstein summation).

In [ ]:
# Two rank-3 tensors (like two MPS sites)
A = np.random.randn(2, 3, 4)   # physical=2, bond_left=3, bond_right=4
B = np.random.randn(4, 3, 2)   # bond_left=4, bond_right=3, physical=2

# Contract the two bond indices (axes 2 of A with 0 of B)
C = np.tensordot(A, B, axes=([2],[0]))   # result shape: (2,3, 3,2)
print("Contracted tensor shape:", C.shape)

## 5. Building a tiny Matrix Product State (MPS) via successive SVDs

**Canonical MPS construction** (the standard way used in TEBD/DMRG)

For a 4-qubit state we will:
1. Reshape the full coefficient tensor → SVD on first cut
2. Reshape the next matrix → SVD on next cut, etc.
3. Store the left-canonical MPS cores.

In [ ]:
def random_mps(N=4, d=2, chi_max=4):
    """Build a random normalized MPS in left-canonical form"""
    tensors = []
    chi = 1
    for n in range(N):
        # Random tensor of shape (χ_left, d, χ_right)
        A = np.random.randn(chi, d, chi_max) + 1j*np.random.randn(chi, d, chi_max)
        A = A[:, :, :chi_max]   # cap bond dim
        
        # Reshape to matrix (χ_left * d) × χ_right
        A_mat = A.reshape(chi*d, -1)
        U, S, Vh = np.linalg.svd(A_mat, full_matrices=False)
        
        # Truncate
        chi_new = min(len(S), chi_max)
        U = U[:, :chi_new]
        S = S[:chi_new]
        Vh = Vh[:chi_new, :]
        
        # Store left-canonical core
        A_can = U.reshape(chi, d, chi_new)
        tensors.append(A_can)
        
        # Update for next site
        chi = chi_new
        # The next matrix will be S @ Vh  (absorbed into next site)
        
    # Last tensor absorbs the remaining singular values (we ignore norm for demo)
    return tensors

mps = random_mps(N=4, chi_max=3)
print("MPS cores shapes:")
for i, t in enumerate(mps):
    print(f"Site {i}: {t.shape}")

**Output**  
Site 0: (1, 2, 3)  
Site 1: (3, 2, 3)  
Site 2: (3, 2, 3)  
Site 3: (3, 2, 1)

This is exactly the data layout you will keep in UPMEM memory banks — each core is tiny and lives next to its own PIM cores.

---

## 6. Basic MPS Contraction: Norm & Local Expectation Value ⟨Z⟩

We will contract the whole network to compute ⟨ψ|ψ⟩ and ⟨Z₀⟩.

In [ ]:
def mps_norm(mps):
    """Naive full contraction for tiny MPS (demo only)"""
    N = len(mps)
    # Start from left
    L = np.array([1.0])  # dummy left bond
    for A in mps:
        L = np.tensordot(L, A, axes=(0,0))      # contract left bond
        L = np.tensordot(L, A.conj(), axes=([0,1],[0,1]))  # contract physical + right bond
    return L.item().real

def mps_expect_Z(mps, site=0):
    """⟨Z⟩ on a given site (Pauli Z = diag(+1,-1))"""
    N = len(mps)
    Z = np.array([[1,0],[0,-1]], dtype=complex)
    
    L = np.array([1.0])
    for i in range(site):
        A = mps[i]
        L = np.tensordot(L, A, axes=(0,0))
        L = np.tensordot(L, A.conj(), axes=([0,1],[0,1]))
    
    # Insert Z on physical leg
    A = mps[site]
    L = np.tensordot(L, A, axes=(0,0))
    L = np.tensordot(L, Z, axes=(1,0))      # contract physical with Z
    L = np.tensordot(L, A.conj(), axes=([0,1],[0,1]))
    
    for i in range(site+1, N):
        A = mps[i]
        L = np.tensordot(L, A, axes=(0,0))
        L = np.tensordot(L, A.conj(), axes=([0,1],[0,1]))
    
    return L.item().real

norm = mps_norm(mps)
z0   = mps_expect_Z(mps, site=0)
print(f"Norm ⟨ψ|ψ⟩ = {norm:.6f}  (should be ~1)")
print(f"⟨Z₀⟩ = {z0:.6f}")

## 7. Next Steps for UPMEM Implementation

- **Memory layout**: Each MPS core lives in its own PIM bank → almost zero data movement for local updates (TEBD sweep).
- **Next notebook ideas**:
  1. TEBD time evolution on UPMEM (tensordot + SVD per bond)
  2. iDMRG / VUMPS for infinite systems
  3. PEPS for 2D (corner-transfer or boundary-MPS contraction)
  4. GPU vs PIM benchmark on the same contraction

**Recommended reading (the two Orús papers you attached)**  
- “A Practical Introduction to Tensor Networks” (2014) – perfect for numerics  
- “Tensor networks for complex quantum systems” (2019) – broader overview + symmetries

Copy the whole notebook above into a fresh `.ipynb` file and run it.  
You now have the minimal working skeleton to start porting to UPMEM!

Happy coding — feel free to ask for the next notebook (TEBD on PIM, PEPS, fermionic TNs, …).